Theme: Vehicle Routing Problem with Time Windows  
Methods: Ant Colony Optimization, PSO, Artificial Bees

In [2]:
import math, random, time
from pathlib import Path
from collections import namedtuple
import numpy as np


In [3]:

# ---------- 0. штрафы и базовые коэффициенты ----------------------------------
BIG       = 100_000            # базовая цена одной машины
P_CAP     = 100_000            # за единицу сверх вместимости
P_LATE    = 10_000             # за минуту опоздания к окну
P_HORIZON = 10_000             # за минуту сверх дневной смены
P_FLEET   = 1_000_000          # за каждую машину сверх K_MAX

# ---------- 1. структура клиента ---------------------------------------------
Customer = namedtuple("Customer",
        "id x y demand ready due service")

# ---------- 2. парсер ---------------------------------------------------------
def parse_solomon(path: str):
    """
    Читает оригинальные benchmark-файлы Solomon/Homberger.
    Возвращает: depot, clients(list), K_max, Q_cap.
    """
    lines = [l.rstrip() for l in Path(path).read_text().splitlines()
             if l.strip()]

    # ---- блок VEHICLE ----
    iv = next(i for i, l in enumerate(lines) if l.upper().startswith("VEHICLE"))
    j  = iv + 1
    while not lines[j].split()[0].isdigit():   # пропускаем заголовок
        j += 1
    K_max, Q_cap = map(int, lines[j].split()[:2])

    # ---- блок CUSTOMER ----
    ic = next(i for i, l in enumerate(lines) if l.upper().startswith("CUSTOMER"))
    rows = [l.split() for l in lines[ic+1:] if l.split()[0].isdigit()]

    d = list(map(float, rows[0]))                              # депо – 1-я строка
    depot = Customer(int(d[0]), d[1], d[2], 0, d[4], d[5], d[6])

    clients = []
    for r in rows[1:]:
        r = list(map(float, r))
        clients.append(Customer(int(r[0]), r[1], r[2],
                                int(r[3]), r[4], r[5], r[6]))
    return depot, clients, K_max, Q_cap


In [4]:
# ---------- 3. Distance counters ------------------------------------
def build_distance_matrix(nodes):
    """Building matrix of distances between all customers."""
    customer_number = len(nodes) # з депо
    distance_matrix = np.zeros((customer_number, customer_number))
    for i in range(customer_number):
        for j in range(customer_number):
            distance_matrix[i, j] = math.hypot(nodes[i].x - nodes[j].x, nodes[i].y - nodes[j].y)
    return distance_matrix

def fast_dist(dist_matrix, a: Customer, b: Customer):
    return dist_matrix[a.id, b.id]

def route_distance(dist_matrix, depot, route):
    """Return DEPOT-CUSTOMER-…CUSTOMER-DEPOT route distance."""
    distance, current_position = 0.0, depot
    for customer in route:
        distance += fast_dist(dist_matrix, current_position, customer)
        current_position = customer
    distance += fast_dist(dist_matrix, current_position, depot)
    return distance


In [5]:
# ---------- 4. Decoder and cost counter -------------------------------------------------
def decode_and_cost(distance_matrix, sequence: list[Customer], depot: Customer, max_vehicles_number: int, vehicle_capacity: int, max_working_time: int):
    routes = [] 
    route = [] 
    load = 0 # нагрузка машини в даний момент
    clock = 0.0 
    current = depot
    penalty = 0.0

    i = 0
    while i < len(sequence):
        customer = sequence[i]
        new_distance = fast_dist(distance_matrix, current, customer)
        arrive = clock + new_distance
        start  = max(arrive, customer.ready)
        finish = start + customer.service

        late = max(0.0, finish - customer.due)
        penalty += late * P_LATE

        time_to_depot = fast_dist(distance_matrix, customer, depot)
        if load + customer.demand > vehicle_capacity or finish + time_to_depot > max_working_time:
            routes.append(route)
            load, clock, current = 0, 0.0, depot
            route = []
            continue  # повторно обробити цього клієнта на новому маршруті

        route.append(customer)
        load  += customer.demand
        clock = finish
        current = customer
        i += 1

    if route:
        routes.append(route)

    # Штраф якщо більше маршрутів(машин) ніж дозволено
    extra_fleet = max(0, len(routes) - max_vehicles_number)
    penalty += extra_fleet * P_FLEET

    km = sum(route_distance(distance_matrix, depot, r) for r in routes)
    cost = len(routes) * BIG + km + penalty
    return routes, cost, km

def fitness(seq, depot, max_vehicles, vehicle_capacity, max_working_time, distance_matrix): 
    return decode_and_cost(distance_matrix, seq, depot, max_vehicles, vehicle_capacity, max_working_time)[1]


In [6]:
class ACO:
    def __init__(self, clients, depot, distance_matrix, K_max, Q_cap, Tmax,
                 n_ants=50, alpha=1, beta=3.0, rho=0.1, n_iter=500):
        self.n_ants, self.alpha, self.beta, self.rho, self.n_iter = \
            n_ants, alpha, beta, rho, n_iter
        self.nodes = clients[:]                # без депо
        self.N = len(self.nodes)

        self.depot = depot
        self.dist_matrix = distance_matrix
        self.K_max = K_max
        self.Q_cap = Q_cap
        self.Tmax = Tmax

        self.tau = np.ones((self.N, self.N))
        self.eta = 1.0 / (np.array([
            [fast_dist(self.dist_matrix, a, b) for b in self.nodes]
            for a in self.nodes]) + 1e-6)

    def run(self):
        best_routes, best_val, best_km = None, float('inf'), float('inf')

        for i in range(self.n_iter):
            tours = [self._build() for _ in range(self.n_ants)]
            for tour in tours:
                routes, v, km = decode_and_cost(
                    self.dist_matrix, tour, self.depot,
                    self.K_max, self.Q_cap, self.Tmax
                )
                if v < best_val:
                    best_val, best_routes, best_km = v, routes, km
            if i % 10 == 0 or i == self.n_iter - 1:
                print(f"Iter {i}: best_val = {best_val:.2f}, best_km = {best_km:.2f}")
            self._update(tours)

        return best_val, best_routes, best_km

    def _build(self):
        remaining = list(range(self.N))
        cur = random.choice(remaining)
        tour = [self.nodes[cur]]
        remaining.remove(cur)
        while remaining:
            probs = (self.tau[cur, remaining] ** self.alpha) * \
                    (self.eta[cur, remaining] ** self.beta)
            probs /= probs.sum()
            nxt = random.choices(remaining, probs)[0]
            tour.append(self.nodes[nxt])
            remaining.remove(nxt)
            cur = nxt
        return tour

    def _update(self, seqs):
        self.tau *= (1 - self.rho)
        for s in seqs:
            v = fitness(s, self.depot, self.K_max, self.Q_cap, self.Tmax, self.dist_matrix)
            for a, b in zip(s[:-1], s[1:]):
                i = self.nodes.index(a)
                j = self.nodes.index(b)
                self.tau[i, j] += 1.0 / v


In [7]:
depot, clients, K, Q = parse_solomon("tests/c101.txt")
dist_matrix = build_distance_matrix([depot] + clients)

aco = ACO(clients, depot, dist_matrix, K, Q, Tmax=depot.due)
best_cost, best_routes, best_km = aco.run()

Iter 0: best_val = 390752938.07, best_km = 1449.81
Iter 10: best_val = 386870969.84, best_km = 1441.40
Iter 20: best_val = 386870969.84, best_km = 1441.40
Iter 30: best_val = 375542182.65, best_km = 1424.92
Iter 40: best_val = 374501041.56, best_km = 1533.49
Iter 50: best_val = 373483475.06, best_km = 1522.03
Iter 60: best_val = 373483475.06, best_km = 1522.03
Iter 70: best_val = 360177383.67, best_km = 1486.13
Iter 80: best_val = 360177383.67, best_km = 1486.13
Iter 90: best_val = 360177383.67, best_km = 1486.13
Iter 100: best_val = 360177383.67, best_km = 1486.13
Iter 110: best_val = 360177383.67, best_km = 1486.13
Iter 120: best_val = 360177383.67, best_km = 1486.13
Iter 130: best_val = 360177383.67, best_km = 1486.13
Iter 140: best_val = 360177383.67, best_km = 1486.13
Iter 150: best_val = 360177383.67, best_km = 1486.13
Iter 160: best_val = 359630650.02, best_km = 1255.49
Iter 170: best_val = 352139447.54, best_km = 1382.62
Iter 180: best_val = 329984315.88, best_km = 1287.73
Iter

KeyboardInterrupt: 

In [8]:
class GA:
    def __init__(self, clients, depot, distance_matrix, K_max, Q_cap, Tmax,
                 pop_size=5, n_gen=2, crossover_rate=0.9, mutation_rate=0.2, tournament_size=3):
        self.nodes = clients[:]
        self.N = len(self.nodes)
        self.depot = depot
        self.dist_matrix = distance_matrix
        self.K_max = K_max
        self.Q_cap = Q_cap
        self.Tmax = Tmax

        self.pop_size = pop_size
        self.n_gen = n_gen
        self.crossover_rate = crossover_rate
        self.mutation_rate = mutation_rate
        self.tournament_size = tournament_size

    def run(self):
        population = [random.sample(self.nodes, self.N) for _ in range(self.pop_size)]
        fitness_cache = []

        for ind in population:
            routes, val, km = decode_and_cost(self.dist_matrix, ind, self.depot,
                                              self.K_max, self.Q_cap, self.Tmax)
            fitness_cache.append((ind, val, km))

        best_ind, best_val, best_km = min(fitness_cache, key=lambda x: x[1])

        for i in range(self.n_gen):
            new_population = []
            while len(new_population) < self.pop_size:
                p1 = self._tournament_selection(fitness_cache)
                p2 = self._tournament_selection(fitness_cache)

                if random.random() < self.crossover_rate:
                    c1, c2 = self._pmx_crossover(p1, p2)
                else:
                    c1, c2 = p1[:], p2[:]

                if random.random() < self.mutation_rate:
                    self._mutate(c1)
                if random.random() < self.mutation_rate:
                    self._mutate(c2)

                new_population.extend([c1, c2])

            population = new_population[:self.pop_size]
            fitness_cache = []
            for ind in population:
                routes, val, km = decode_and_cost(self.dist_matrix, ind, self.depot,
                                                  self.K_max, self.Q_cap, self.Tmax)
                fitness_cache.append((ind, val, km))

            current_best = min(fitness_cache, key=lambda x: x[1])
            if current_best[1] < best_val:
                best_ind, best_val, best_km = current_best

            if i % 10 == 0 or i == self.n_gen - 1:
                print(f"Gen {i}: best_val = {best_val:.2f}, best_km = {best_km:.2f}")

        best_routes, _, _ = decode_and_cost(self.dist_matrix, best_ind, self.depot,
                                            self.K_max, self.Q_cap, self.Tmax)
        return best_val, best_routes, best_km

    def _tournament_selection(self, fitness_cache):
        selected = random.sample(fitness_cache, self.tournament_size)
        selected.sort(key=lambda x: x[1])
        return selected[0][0][:]

    def _pmx_crossover(self, p1, p2):
        size = len(p1)
        c1, c2 = [None]*size, [None]*size
        a, b = sorted(random.sample(range(size), 2))

        c1[a:b+1] = p1[a:b+1]
        c2[a:b+1] = p2[a:b+1]

        def pmx_fill(c, p, p_base):
            index_map = {v: i for i, v in enumerate(p_base)}
            for i in range(a, b+1):
                if p[i] not in c:
                    pos = i
                    while c[pos] is not None:
                        pos = index_map[p_base[pos]]
                    c[pos] = p[i]
            for i in range(size):
                if c[i] is None:
                    c[i] = p[i]

        pmx_fill(c1, p2, p1)
        pmx_fill(c2, p1, p2)
        return c1, c2

    def _mutate(self, individual):
        i, j = random.sample(range(len(individual)), 2)
        individual[i], individual[j] = individual[j], individual[i]

In [ ]:
depot, clients, K, Q = parse_solomon("tests/c101.txt")
dist_matrix = build_distance_matrix([depot] + clients)


In [9]:
ga = GA(clients, depot, dist_matrix, K, Q, Tmax=depot.due)
best_cost, best_routes, best_km = ga.run()

KeyboardInterrupt: 

In [ ]:
depot.due

1236.0